# Abstraction Review

In this notebook, we will demonstrate a simple example of how LatteReview could be used for abstracting concepts from input data.

## Setting up the notebook

High-level configs

In [1]:
%reload_ext autoreload
%autoreload 2

from dotenv import load_dotenv

# Load environment variables from .env file. Adjust the path to the .env file as needed.
load_dotenv(dotenv_path='../../.env')

# Enable asyncio in Jupyter
import asyncio
import nest_asyncio

nest_asyncio.apply()

#  Add the package to the path (required if you are running this notebook from the examples folder)
import sys
sys.path.append('../../')


Import required packages

In [2]:
import json
import pandas as pd
from pydantic import BaseModel
from tqdm.auto import tqdm

from lattereview.providers import OpenAIProvider
from lattereview.providers import LiteLLMProvider
from lattereview.agents import AbstractionReviewer
from lattereview.workflows import ReviewWorkflow

## Data

To demonstrate how the `AbstractionReviewer` can be used to extract specific information from input items, we will create a dummy dataset of imaginary stories. The `AbstractionReviewer` will be tasked with extracting the location of each story and identifying the main characters introduced in it. To generate these stories, we will prompt the GPT-6 Sol model to provide the stories along with the ground truth location and characters. Notice how we utilize the base functionalities of LatteLLM and a Pydantic base model to specify the expected output format from GPT-6 Sol, clearly defining the structure of the desired output.

In [3]:
class BuildStoryOutput(BaseModel):
    story: str
    location: str
    characters: list[str]

async def build_story():
    prompt = """
    Write a one-paragraph story with whatever realistic or imaginary theme you like,  
    then create a list of all characters you named in your story.
    Return your story, the main location that your story happens in, and a Python list of your characters as your output.
    """
    provider = OpenAIProvider(model="gpt-6-sol", response_format_class=BuildStoryOutput)
    return await provider.get_json_response(prompt)

def run_build_story():
    response =  asyncio.run(build_story())[0]
    return response

data = {
    "story": [],
    "location": [],
    "characters": [],
}
for i in tqdm(range(5)):
    out = json.loads(run_build_story())
    data["characters"].append(out["characters"])
    data["location"].append(out["location"])
    data["story"].append(out["story"])


data = pd.DataFrame(data)
data.to_csv("data.csv", index=False)
data

100%|██████████| 5/5 [00:20<00:00,  4.10s/it]


,story,location,characters
0,"Every evening, Mara tended the lighthouse on a...",A lighthouse on a dry seabed,[Mara]
1,"Every evening, Mara tended the lighthouse on B...",Bellweather Island lighthouse,"[Mara, Elias]"
2,"Every evening, Mara climbed the lighthouse sta...",The lighthouse at the harbor,"[Mara, Ivo]"
3,"At the old lighthouse on Gull Island, Mara fou...",The old lighthouse on Gull Island,"[Mara, Eli, Jun]"
4,"At Bellwether Lighthouse, Mira found a glass b...",Bellwether Lighthouse,"[Mira, Tomas]"


## Abstraction with a single agent

Here, we will use the `AbstractionReviewer` without defining a workflow. To do so, we simply need to specify the keys we expect the `AbstractionReviewer` to output, which in this case are the location and characters keys. Additionally, we need to define the meaning of each key in the `key_descriptions` argument.

In [4]:
Albert = AbstractionReviewer(
    provider=LiteLLMProvider(model="gpt-6-luna"),
    name="Albert",
    max_concurrent_requests=1, 
    backstory="an expert reviewer!",
    input_description = "stories",
    model_args={"reasoning_effort": "low"},
    abstraction_keys = {
        "location": str, 
        "characters": list[str]
    },
    key_descriptions = {
        "location": "The main location that the story happens in.", 
        "characters": "The name of the characters mentioned in the story."
    }
)


# Dummy input
input_list = data.story.str.lower().tolist()
print("====== Inputs ======\n\n", '\n'.join(input_list))

# Dummy review
results, total_cost = asyncio.run(Albert.review_items(input_list))
print("\n====== Outputs ======")
for result in results:
    print(result)

# Dummy costs
print("\n====== Costs ======\n")
for i, item in enumerate(Albert.memory):
    print(f"Cost for item {i}: {item['cost']}")

print(f"\nTotal cost: {total_cost}")

====== Inputs ======

 every evening, mara tended the lighthouse on an island where the sea had vanished years ago. one night, its beam caught a glimmer on the horizon: a wave, no taller than a hand, rolling slowly across the dry seabed. mara filled a cup from the kitchen tap and poured it into the wave. by morning, the tide had reached the lighthouse steps, and the fishing boats lay afloat once more.
every evening, mara tended the lighthouse on bellweather island, though ships had stopped passing years ago. one foggy night, a small boat appeared below the cliffs, and its passenger, elias, climbed the path carrying a sealed letter. it was from mara’s brother, lost at sea a decade earlier, and contained only four words: “keep the light burning.” mara looked toward the harbor and saw, far beyond the fog, the first fishing boat returning home.
every evening, mara climbed the lighthouse stairs to wind its lamp, though no ships had visited the harbor in years. one foggy night, a small boat 

Reviewing 5 items - 2026-09-27 13:30:06: 100%|██████████| 5/5 [00:05<00:00,  1.06s/it]


====== Outputs ======
{'location': 'An island with a lighthouse beside a dry seabed', 'characters': ['Mara']}
{'location': 'Bellweather Island', 'characters': ['Mara', 'Elias']}
{'location': 'A harbor town, centered around its lighthouse', 'characters': ['Mara', 'Ivo']}
{'location': 'The old lighthouse on Gull Island', 'characters': ['Mara', 'Eli', 'Jun']}
{'location': 'Bellwether lighthouse', 'characters': ['Mira', 'Tomas']}

====== Costs ======

Cost for item 0: 4.31e-05
Cost for item 1: 4.22e-05
Cost for item 2: 4.3299999999999995e-05
Cost for item 3: 4.42e-05
Cost for item 4: 4.0499999999999995e-05

Total cost: 0.00021329999999999998


## Abstraction with a workflow

Obviously, the same functionality could also be achieved by defining a workflow. In this case, we will define a workflow with a single item and a single agent to demonstrate how AbstractionReviewers are similar to ScoringReviewers when incorporated into workflows. Naturally, in more complex reviews, an AbstractionReviewer could be combined with other AbstractionReviewers or even ScoringReviewers to accomplish more sophisticated review goals.

In [5]:
workflow = ReviewWorkflow(
    workflow_schema=[
        {
            "round": 'A',
            "reviewers": [Albert],
            "text_inputs": ["story"]
        }
    ]
)

# Reload the data if needed.
updated_data = asyncio.run(workflow(data))

print("\n====== Costs ======\n")
print("Total cost: ", workflow.get_total_cost())
print("Detailed costs: ", workflow.reviewer_costs)

updated_data


====== Starting review round A (1/1) ======

Processing 5 eligible rows


['round: A', 'reviewer_name: Albert'] -                     2026-09-27 13:30:12: 100%|██████████| 5/5 [00:04<00:00,  1.11it/s]

Reviewer Albert: 5 successful, 0 failed parses
The following columns are present in the dataframe at the end of Albert's review in round A: ['story', 'location', 'characters', 'round-A_Albert_output', 'round-A_Albert_location', 'round-A_Albert_characters']

====== Costs ======

Total cost:  0.00021370000000000002
Detailed costs:  {('A', 'Albert'): 0.00021370000000000002}


,story,location,characters,round-A_Albert_output,round-A_Albert_location,round-A_Albert_characters
0,"Every evening, Mara tended the lighthouse on a...",A lighthouse on a dry seabed,[Mara],"{'location': 'A lighthouse on an island', 'cha...",A lighthouse on an island,[Mara]
1,"Every evening, Mara tended the lighthouse on B...",Bellweather Island lighthouse,"[Mara, Elias]","{'location': 'Bellweather Island', 'characters...",Bellweather Island,"[Mara, Elias]"
2,"Every evening, Mara climbed the lighthouse sta...",The lighthouse at the harbor,"[Mara, Ivo]","{'location': 'A lighthouse and harbor', 'chara...",A lighthouse and harbor,"[Mara, Ivo]"
3,"At the old lighthouse on Gull Island, Mara fou...",The old lighthouse on Gull Island,"[Mara, Eli, Jun]",{'location': 'The old lighthouse on Gull Islan...,The old lighthouse on Gull Island,"[Mara, Eli, Jun]"
4,"At Bellwether Lighthouse, Mira found a glass b...",Bellwether Lighthouse,"[Mira, Tomas]","{'location': 'Bellwether Lighthouse', 'charact...",Bellwether Lighthouse,"[Mira, Tomas]"


In [6]:
Albert.formatted_prompt

"**Review the input item below and extract the specified keys as instructed:** --- **Input Item:** <<${item}$>> **Keys to Extract and Their Expected Formats:** <<{'location': <class 'str'>, 'characters': list[str]}>> --- **Instructions:** Follow the detailed guidelines below for extracting the specified keys: <<{'location': 'The main location that the story happens in.', 'characters': 'The name of the characters mentioned in the story.'}>> --- ${additional_context}$"